In [ ]:
# @title **CELDA 5: PROPUESTAS DE MEJORA Y PLAN DE ACCIÓN**

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import re
import warnings
warnings.filterwarnings('ignore')

print("🚀 CELDA 5: PROPUESTAS DE MEJORA Y PLAN DE ACCIÓN")
print("="*80)
print("OBJETIVO: Generar recomendaciones basadas en el análisis FIFO y crear plan de acción")
print("="*80)

# ===============================
# CONFIGURACIÓN
# ===============================
INPUT_DIR = "input"
OUTPUT_DIR = "output"

# Configurar estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ===============================
# 1. ANALIZAR RESULTADOS FINALES
# ===============================
print("\n🔍 1. Analizando resultados finales del FIFO Físico...")

# Cargar datos finales
df_tabla_final = pd.read_csv(f"{OUTPUT_DIR}/Tabla_Final_FIFO.csv")
df_log_cambios = pd.read_csv(f"{OUTPUT_DIR}/Log_Cambios_FIFO_Detallado.csv")
df_resumen_ejecutivo = pd.read_csv(f"{OUTPUT_DIR}/Resumen_Ejecutivo_FIFO.csv")
df_errores_originales = pd.read_csv(f"{INPUT_DIR}/Log_Errores_Ingresos.csv")

# Calcular métricas clave
total_residuos = len(df_tabla_final)
residuos_ajustados = len(df_tabla_final[df_tabla_final["Diferencia_Ajustada_kg"].abs() <= 0.01])
porcentaje_ajustados = (residuos_ajustados / total_residuos) * 100

print(f"   📊 Métricas finales:")
print(f"      • Residuos analizados: {total_residuos}")
print(f"      • Residuos completamente ajustados: {residuos_ajustados} ({porcentaje_ajustados:.1f}%)")
print(f"      • Ajustes aplicados: {len(df_log_cambios)}")
print(f"      • Error total corregido: {df_resumen_ejecutivo['Error_Total_Corregido_kg'].iloc[0]:,.0f} kg")

# ===============================
# 2. IDENTIFICAR PATRONES DE ERRORES
# ===============================
print("\n📉 2. Identificando patrones de errores recurrentes...")

# 2.1 Análisis de tipos de errores originales
if not df_errores_originales.empty:
    print(f"\n   🔍 Tipos de errores encontrados en los datos originales:")
    errores_por_tipo = df_errores_originales["Tipo_Error"].value_counts()

    for tipo, cantidad in errores_por_tipo.items():
        porcentaje = (cantidad / len(df_errores_originales)) * 100
        error_promedio = df_errores_originales[df_errores_originales["Tipo_Error"] == tipo]["Error_kg"].abs().mean()
        print(f"      • {tipo}: {cantidad} eventos ({porcentaje:.1f}%), Error promedio: {error_promedio:.0f} kg")

# 2.2 Análisis de residuos con mayores discrepancias originales
print(f"\n   📊 Residuos con mayores discrepancias originales:")
top_discrepancias = df_tabla_final.nlargest(5, "Diferencia_Original_kg", "all")
for idx, row in top_discrepancias.iterrows():
    print(f"      • {row['Residuo'][:25]}: {row['Diferencia_Original_kg']:+,.0f} kg")

# 2.3 Análisis por tipo de residuo
print(f"\n   📈 Análisis por tipo de residuo:")
tipos_residuo = df_tabla_final.groupby("Tipo").agg({
    "Residuo": "count",
    "Diferencia_Original_kg": ["sum", "mean"],
    "Mejora_kg": "sum"
}).round(2)

for tipo, datos in tipos_residuo.iterrows():
    print(f"      • {tipo}: {datos[('Residuo', 'count')]} residuos, "
          f"Diferencia: {datos[('Diferencia_Original_kg', 'sum')]:,.0f} kg, "
          f"Mejora: {datos[('Mejora_kg', 'sum')]:,.0f} kg")

# ===============================
# 3. GENERAR PROPUESTAS DE MEJORA
# ===============================
print("\n💡 3. Generando propuestas de mejora basadas en hallazgos...")

propuestas = []

# 3.1 Propuestas tecnológicas
propuestas.append({
    "Categoria": "TECNOLÓGICA",
    "Prioridad": "ALTA",
    "Propuesta": "Sistema de Pesaje Automatizado con IoT",
    "Descripcion": "Implementar básculas inteligentes conectadas a sistema central",
    "Beneficios": [
        "Reducción del 95% en errores de pesaje manual",
        "Integración automática con sistema de gestión",
        "Alertas en tiempo real para desviaciones"
    ],
    "Inversion_Estimada_USD": 50000,
    "ROI_Estimado": 2.5,
    "Plazo_Implementacion": "6-9 meses",
    "Responsable": "Gerencia de Tecnología",
    "Metricas_Exito": ["Error de pesaje < 1%", "Tiempo registro reducido en 70%"]
})

propuestas.append({
    "Categoria": "TECNOLÓGICA",
    "Prioridad": "MEDIA",
    "Propuesta": "Sistema de Códigos QR/RFID para Contenedores",
    "Descripcion": "Identificación única y trazabilidad completa de contenedores",
    "Beneficios": [
        "Eliminación de errores de clasificación",
        "Trazabilidad en tiempo real",
        "Control de movimientos automatizado"
    ],
    "Inversion_Estimada_USD": 25000,
    "ROI_Estimado": 3.0,
    "Plazo_Implementacion": "4-6 meses",
    "Responsable": "Gerencia de Operaciones",
    "Metricas_Exito": ["100% contenedores identificados", "Errores clasificación reducidos en 90%"]
})

# 3.2 Propuestas de procesos
propuestas.append({
    "Categoria": "PROCESOS",
    "Prioridad": "ALTA",
    "Propuesta": "Auditorías FIFO Periódicas Trimestrales",
    "Descripcion": "Implementar auditorías sistemáticas aplicando método FIFO Físico",
    "Beneficios": [
        "Detección temprana de discrepancias",
        "Prevención de balances negativos",
        "Mejora continua en precisión de inventarios"
    ],
    "Inversion_Estimada_USD": 10000,
    "ROI_Estimado": 4.0,
    "Plazo_Implementacion": "1 mes",
    "Responsable": "Control de Gestión",
    "Metricas_Exito": ["Discrepancias < 1%", "0 balances negativos"]
})

propuestas.append({
    "Categoria": "PROCESOS",
    "Prioridad": "ALTA",
    "Propuesta": "Protocolo de Verificación de Cambios de Clasificación",
    "Descripcion": "Procedimiento formal para registrar y validar cambios en clasificación de residuos",
    "Beneficios": [
        "Eliminación de errores por reclasificación no registrada",
        "Consistencia en datos de inventario",
        "Mejor cumplimiento normativo"
    ],
    "Inversion_Estimada_USD": 5000,
    "ROI_Estimado": 5.0,
    "Plazo_Implementacion": "2 meses",
    "Responsable": "Gerencia Ambiental",
    "Metricas_Exito": ["100% cambios registrados", "0 errores por reclasificación"]
})

propuestas.append({
    "Categoria": "PROCESOS",
    "Prioridad": "MEDIA",
    "Propuesta": "Checklist de Verificación de Salidas",
    "Descripcion": "Lista de verificación obligatoria antes de autorizar salidas de residuos",
    "Beneficios": [
        "Reducción de errores en destino y cantidad",
        "Documentación completa de cada salida",
        "Mejor trazabilidad para auditorías"
    ],
    "Inversion_Estimada_USD": 3000,
    "ROI_Estimado": 6.0,
    "Plazo_Implementacion": "1 mes",
    "Responsable": "Supervisor de Almacén",
    "Metricas_Exito": ["100% salidas verificadas", "Errores salida reducidos en 80%"]
})

# 3.3 Propuestas de capacitación
propuestas.append({
    "Categoria": "CAPACITACIÓN",
    "Prioridad": "ALTA",
    "Propuesta": "Programa de Certificación en Gestión FIFO",
    "Descripcion": "Curso teórico-práctico en método FIFO y gestión precisa de inventarios",
    "Beneficios": [
        "Personal capacitado en mejores prácticas",
        "Reducción de errores humanos",
        "Cultura de precisión en datos"
    ],
    "Inversion_Estimada_USD": 15000,
    "ROI_Estimado": 3.5,
    "Plazo_Implementacion": "3 meses",
    "Responsable": "RRHH y Gerencia de Operaciones",
    "Metricas_Exito": ["100% personal certificado", "Errores humanos reducidos en 70%"]
})

propuestas.append({
    "Categoria": "CAPACITACIÓN",
    "Prioridad": "MEDIA",
    "Propuesta": "Talleres de Concientización Ambiental",
    "Descripcion": "Sesiones sobre importancia del registro preciso para cumplimiento ambiental",
    "Beneficios": [
        "Mayor compromiso del personal",
        "Mejor entendimiento de impactos",
        "Reducción de riesgos regulatorios"
    ],
    "Inversion_Estimada_USD": 8000,
    "ROI_Estimado": 2.0,
    "Plazo_Implementacion": "4 meses",
    "Responsable": "Gerencia Ambiental",
    "Metricas_Exito": ["100% asistencia", "Mejora en actitud hacia registros"]
})

# 3.4 Propuestas de monitoreo y control
propuestas.append({
    "Categoria": "MONITOREO",
    "Prioridad": "ALTA",
    "Propuesta": "Dashboard de Gestión en Tiempo Real",
    "Descripcion": "Panel de control con métricas clave actualizadas automáticamente",
    "Beneficios": [
        "Visibilidad inmediata de problemas",
        "Toma de decisiones basada en datos",
        "Alertas proactivas para desviaciones"
    ],
    "Inversion_Estimada_USD": 20000,
    "ROI_Estimado": 4.5,
    "Plazo_Implementacion": "5 meses",
    "Responsable": "Gerencia de Tecnología",
    "Metricas_Exito": ["Acceso 24/7 a métricas", "Tiempo detección problemas reducido en 90%"]
})

propuestas.append({
    "Categoria": "MONITOREO",
    "Prioridad": "MEDIA",
    "Propuesta": "Sistema de Alertas Automáticas",
    "Descripcion": "Alertas configuradas para discrepancias mayores a umbrales definidos",
    "Beneficios": [
        "Respuesta inmediata a problemas",
        "Prevención de errores acumulativos",
        "Reducción de trabajo correctivo"
    ],
    "Inversion_Estimada_USD": 12000,
    "ROI_Estimado": 3.8,
    "Plazo_Implementacion": "3 meses",
    "Responsable": "Control de Gestión",
    "Metricas_Exito": ["Alertas en < 1 hora", "0 discrepancias > 100 kg no detectadas"]
})

# Convertir a DataFrame
df_propuestas = pd.DataFrame(propuestas)

# ===============================
# 4. PRIORIZACIÓN Y ROADMAP
# ===============================
print("\n🎯 4. Priorizando propuestas y creando roadmap...")

# 4.1 Función para extraer meses de plazo
def extraer_meses_plazo(plazo_str):
    """Extrae el número de meses de una cadena de texto de plazo"""
    # Buscar números en la cadena
    numeros = re.findall(r'\d+', str(plazo_str))
    if numeros:
        # Tomar el primer número encontrado
        return int(numeros[0])
    return 12  # Valor por defecto si no se encuentra número

# 4.2 Calcular puntaje de prioridad
def calcular_puntaje_prioridad(row):
    puntaje = 0

    # Prioridad: ALTA=3, MEDIA=2, BAJA=1
    prioridad_map = {"ALTA": 3, "MEDIA": 2, "BAJA": 1}
    puntaje += prioridad_map.get(row["Prioridad"], 1)

    # ROI: mayor ROI = mayor puntaje (escalado)
    puntaje += min(row["ROI_Estimado"], 5)  # Máximo 5 puntos por ROI

    # Plazo: menor plazo = mayor puntaje
    plazo_meses = extraer_meses_plazo(row["Plazo_Implementacion"])
    if plazo_meses <= 3:
        puntaje += 3
    elif plazo_meses <= 6:
        puntaje += 2
    else:
        puntaje += 1

    return puntaje

# Aplicar función de puntaje
df_propuestas["Puntaje_Prioridad"] = df_propuestas.apply(calcular_puntaje_prioridad, axis=1)
df_propuestas["Ranking"] = df_propuestas["Puntaje_Prioridad"].rank(method="dense", ascending=False).astype(int)

# 4.3 Crear fases de implementación
def asignar_fase(row):
    if row["Ranking"] <= 3:
        return "FASE 1: Corto Plazo (0-3 meses)"
    elif row["Ranking"] <= 6:
        return "FASE 2: Mediano Plazo (3-6 meses)"
    else:
        return "FASE 3: Largo Plazo (6-12 meses)"

df_propuestas["Fase_Implementacion"] = df_propuestas.apply(asignar_fase, axis=1)

# 4.4 Ordenar por prioridad
df_propuestas = df_propuestas.sort_values(["Ranking", "Puntaje_Prioridad"], ascending=[True, False])

print(f"   ✅ Propuestas priorizadas: {len(df_propuestas)}")
print(f"   📅 Distribución por fases:")
for fase, grupo in df_propuestas.groupby("Fase_Implementacion"):
    print(f"      • {fase}: {len(grupo)} propuestas")

# ===============================
# 5. CREAR PLAN DE ACCIÓN DETALLADO
# ===============================
print("\n📋 5. Creando plan de acción detallado...")

# 5.1 Plan para Fase 1 (Corto Plazo)
plan_accion_fase1 = df_propuestas[df_propuestas["Fase_Implementacion"] == "FASE 1: Corto Plazo (0-3 meses)"].copy()

# 5.2 Agregar hitos específicos
def generar_hitos(row):
    if "Auditorías" in row["Propuesta"]:
        return ["Designar equipo auditor", "Desarrollar checklist", "Ejecutar primera auditoría"]
    elif "Protocolo" in row["Propuesta"]:
        return ["Documentar procedimiento", "Validar con equipos", "Implementar en sistema"]
    elif "Checklist" in row["Propuesta"]:
        return ["Diseñar checklist", "Capacitar personal", "Implementar en operaciones"]
    elif "Certificación" in row["Propuesta"]:
        return ["Desarrollar contenido", "Programar sesiones", "Evaluar competencias"]
    else:
        return ["Análisis de requerimientos", "Selección de proveedores", "Implementación piloto"]

df_propuestas["Hitos_Clave"] = df_propuestas.apply(generar_hitos, axis=1)

# 5.3 Calcular presupuesto por fase
presupuesto_por_fase = df_propuestas.groupby("Fase_Implementacion").agg({
    "Propuesta": "count",
    "Inversion_Estimada_USD": "sum",
    "ROI_Estimado": "mean"
}).round(2)

print(f"\n   💰 PRESUPUESTO ESTIMADO POR FASE:")
for fase, datos in presupuesto_por_fase.iterrows():
    print(f"      • {fase}: {datos['Propuesta']} propuestas, "
          f"USD {datos['Inversion_Estimada_USD']:,.0f}, ROI promedio: {datos['ROI_Estimado']:.1f}x")

# ===============================
# 6. VISUALIZACIÓN DEL ROADMAP
# ===============================
print("\n📊 6. Generando visualización del roadmap...")

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('ROADMAP DE IMPLEMENTACIÓN - PROPUESTAS DE MEJORA', fontsize=16, fontweight='bold')

# 6.1 Gráfico 1: Distribución por categoría y fase
ax1 = axes[0, 0]
categoria_fase_counts = pd.crosstab(df_propuestas["Categoria"], df_propuestas["Fase_Implementacion"])
categoria_fase_counts.plot(kind='bar', ax=ax1, color=['lightblue', 'lightgreen', 'lightcoral'])
ax1.set_title('Distribución de Propuestas por Categoría y Fase', fontsize=12, fontweight='bold')
ax1.set_xlabel('Categoría')
ax1.set_ylabel('Número de Propuestas')
ax1.legend(title='Fase de Implementación')
ax1.grid(True, alpha=0.3, axis='y')

# 6.2 Gráfico 2: ROI vs Inversión
ax2 = axes[0, 1]
scatter = ax2.scatter(df_propuestas["Inversion_Estimada_USD"],
                      df_propuestas["ROI_Estimado"],
                      c=df_propuestas["Puntaje_Prioridad"],
                      s=150, alpha=0.7, cmap='viridis')
ax2.set_title('ROI vs Inversión (Tamaño: Prioridad)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Inversión Estimada (USD)')
ax2.set_ylabel('ROI Estimado (x)')
ax2.grid(True, alpha=0.3)

# Añadir etiquetas para las top 3 propuestas
for idx, row in df_propuestas.head(3).iterrows():
    ax2.annotate(row["Propuesta"][:15] + "...",
                (row["Inversion_Estimada_USD"], row["ROI_Estimado"]),
                textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

# 6.3 Gráfico 3: Timeline simplificado
ax3 = axes[1, 0]
ax3.axis('off')

# Crear timeline visual
fases = {
    "FASE 1: Corto Plazo (0-3 meses)": 1,
    "FASE 2: Mediano Plazo (3-6 meses)": 2,
    "FASE 3: Largo Plazo (6-12 meses)": 3
}

y_pos = 0
for fase, num_fase in fases.items():
    propuestas_fase = df_propuestas[df_propuestas["Fase_Implementacion"] == fase]

    # Dibujar línea de tiempo
    ax3.add_patch(Rectangle((num_fase-0.4, y_pos-0.1), 0.8, 0.2,
                           facecolor='lightblue', alpha=0.5))
    ax3.text(num_fase, y_pos, f"{len(propuestas_fase)} propuestas",
            ha='center', va='center', fontsize=10, fontweight='bold')

    # Listar propuestas
    for i, (_, prop) in enumerate(propuestas_fase.iterrows()):
        ax3.text(num_fase, y_pos - (i+1)*0.3, f"• {prop['Propuesta'][:20]}...",
                ha='center', va='center', fontsize=8)

    y_pos -= (len(propuestas_fase) + 1) * 0.3 + 0.5

ax3.set_xlim(0, 4)
ax3.set_ylim(y_pos, 1)
ax3.set_title('Timeline de Implementación por Fase', fontsize=12, fontweight='bold', pad=20)

# 6.4 Gráfico 4: Matriz de Impacto vs Esfuerzo
ax4 = axes[1, 1]

# Función para extraer meses de implementación
def extraer_meses_implem(plazo_str):
    """Extrae el número de meses de implementación"""
    numeros = re.findall(r'\d+', str(plazo_str))
    if numeros:
        return int(numeros[0])
    return 6  # Valor por defecto

# Calcular esfuerzo (basado en inversión y plazo)
def calcular_esfuerzo(row):
    inversion_norm = row["Inversion_Estimada_USD"] / 50000  # Normalizar a 50k
    plazo_meses = extraer_meses_implem(row["Plazo_Implementacion"])
    plazo_norm = plazo_meses / 12  # Normalizar a 12 meses
    return (inversion_norm + plazo_norm) / 2

def calcular_impacto(row):
    impacto = 0
    impacto += row["ROI_Estimado"] / 5  # ROI máximo 5
    impacto += 0.3 if row["Prioridad"] == "ALTA" else 0.2 if row["Prioridad"] == "MEDIA" else 0.1
    # Contar beneficios
    beneficios_count = len(row["Beneficios"]) if isinstance(row["Beneficios"], list) else 0
    impacto += beneficios_count * 0.1
    return min(impacto, 1.0)  # Limitar a 1.0

# Aplicar cálculos
df_propuestas["Esfuerzo_Relativo"] = df_propuestas.apply(calcular_esfuerzo, axis=1)
df_propuestas["Impacto_Relativo"] = df_propuestas.apply(calcular_impacto, axis=1)

# Crear scatter plot
scatter = ax4.scatter(df_propuestas["Esfuerzo_Relativo"],
                      df_propuestas["Impacto_Relativo"],
                      c=df_propuestas["Puntaje_Prioridad"],
                      s=200, alpha=0.7, cmap='coolwarm')

# Dividir en cuadrantes
ax4.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax4.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

# Etiquetar cuadrantes
ax4.text(0.25, 0.75, 'ALTO IMPACTO\nBAJO ESFUERZO', ha='center', va='center',
        fontsize=10, fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
ax4.text(0.75, 0.75, 'ALTO IMPACTO\nALTO ESFUERZO', ha='center', va='center',
        fontsize=10, fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
ax4.text(0.25, 0.25, 'BAJO IMPACTO\nBAJO ESFUERZO', ha='center', va='center',
        fontsize=10, fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
ax4.text(0.75, 0.25, 'BAJO IMPACTO\nALTO ESFUERZO', ha='center', va='center',
        fontsize=10, fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

ax4.set_title('Matriz Impacto vs Esfuerzo', fontsize=12, fontweight='bold')
ax4.set_xlabel('Esfuerzo Relativo')
ax4.set_ylabel('Impacto Relativo')
ax4.grid(True, alpha=0.3)

# Ajustar layout
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/Roadmap_Propuestas_Mejora.png', dpi=300, bbox_inches='tight')
print(f"   ✅ Roadmap guardado en: {OUTPUT_DIR}/Roadmap_Propuestas_Mejora.png")

# ===============================
# 7. GUARDAR PROPUESTAS Y PLAN DE ACCIÓN
# ===============================
print("\n💾 7. Guardando propuestas y plan de acción...")

# 7.1 Guardar propuestas completas
# Convertir listas a strings para guardar en CSV
df_propuestas_guardar = df_propuestas.copy()
for col in ["Beneficios", "Metricas_Exito", "Hitos_Clave"]:
    df_propuestas_guardar[col] = df_propuestas_guardar[col].apply(
        lambda x: "; ".join(x) if isinstance(x, list) else str(x)
    )

df_propuestas_guardar.to_csv(f"{OUTPUT_DIR}/Propuestas_Mejora_Completas.csv", index=False, encoding='utf-8-sig')

# 7.2 Plan de acción ejecutivo (solo Fase 1)
plan_accion_ejecutivo = df_propuestas[df_propuestas["Ranking"] <= 5].copy()

# Convertir columnas de listas a strings
for col in ["Beneficios", "Metricas_Exito", "Hitos_Clave"]:
    plan_accion_ejecutivo[col] = plan_accion_ejecutivo[col].apply(
        lambda x: "; ".join(x) if isinstance(x, list) else str(x)
    )

plan_accion_ejecutivo = plan_accion_ejecutivo[[
    "Ranking", "Propuesta", "Descripcion", "Beneficios",
    "Inversion_Estimada_USD", "ROI_Estimado", "Plazo_Implementacion",
    "Responsable", "Metricas_Exito", "Hitos_Clave"
]]
plan_accion_ejecutivo.to_csv(f"{OUTPUT_DIR}/Plan_Accion_Ejecutivo.csv", index=False, encoding='utf-8-sig')

# 7.3 Resumen por categoría
resumen_categoria = df_propuestas.groupby("Categoria").agg({
    "Propuesta": "count",
    "Inversion_Estimada_USD": "sum",
    "ROI_Estimado": "mean",
    "Puntaje_Prioridad": "mean"
}).round(2)
resumen_categoria.to_csv(f"{OUTPUT_DIR}/Resumen_por_Categoria.csv", encoding='utf-8-sig')

# 7.4 Reporte ejecutivo final
reporte_final = f"""
{'='*100}
REPORTE FINAL - PROPUESTAS DE MEJORA BASADAS EN ANÁLISIS FIFO FÍSICO
{'='*100}

1. CONTEXTO Y ANTECEDENTES:
   • Ejercicio FIFO Físico completado exitosamente
   • {residuos_ajustados} de {total_residuos} residuos ajustados ({porcentaje_ajustados:.1f}%)
   • {len(df_log_cambios)} ajustes aplicados para corregir {df_resumen_ejecutivo['Error_Total_Corregido_kg'].iloc[0]:,.0f} kg de error

2. HALLAZGOS PRINCIPALES:
"""

# Añadir hallazgos de errores
if not df_errores_originales.empty:
    errores_top = errores_por_tipo.head(3)
    reporte_final += f"   • Principales tipos de errores identificados: {', '.join(errores_top.index.tolist())}\n"
else:
    reporte_final += "   • No se encontraron errores registrados en los datos originales\n"

reporte_final += f"   • Residuos con mayores discrepancias: {', '.join(top_discrepancias['Residuo'].head(3).tolist())}\n"
reporte_final += f"   • Inversión estimada requerida: USD {presupuesto_por_fase['Inversion_Estimada_USD'].sum():,.0f}\n"
reporte_final += f"   • ROI promedio esperado: {presupuesto_por_fase['ROI_Estimado'].mean():.1f}x\n"

reporte_final += f"""
3. PROPUESTAS DE MEJORA PRIORIZADAS:
   Total propuestas generadas: {len(df_propuestas)}

   FASE 1 - CORTO PLAZO (0-3 meses):
"""

# Añadir propuestas de Fase 1
fase1_propuestas = df_propuestas[df_propuestas["Fase_Implementacion"] == "FASE 1: Corto Plazo (0-3 meses)"]
for idx, prop in fase1_propuestas.iterrows():
    reporte_final += f"""
   • {prop['Propuesta']}
     Categoría: {prop['Categoria']} | Prioridad: {prop['Prioridad']}
     Inversión: USD {prop['Inversion_Estimada_USD']:,.0f} | ROI: {prop['ROI_Estimado']:.1f}x
     Responsable: {prop['Responsable']}
"""

reporte_final += f"""
4. BENEFICIOS ESPERADOS:
   • Reducción del 90-95% en errores de registro
   • Eliminación de balances negativos
   • Mejora del 80% en trazabilidad de residuos
   • Ahorro anual estimado: USD {presupuesto_por_fase['Inversion_Estimada_USD'].sum() * presupuesto_por_fase['ROI_Estimado'].mean() / 1000:.1f}K
   • Cumplimiento normativo mejorado

5. PLAN DE IMPLEMENTACIÓN RECOMENDADO:
"""

# Añadir plan por fases
for fase, grupo in df_propuestas.groupby("Fase_Implementacion"):
    reporte_final += f"""
   {fase}:
     • {len(grupo)} propuestas
     • Inversión: USD {grupo['Inversion_Estimada_USD'].sum():,.0f}
     • ROI promedio: {grupo['ROI_Estimado'].mean():.1f}x
"""

reporte_final += f"""
6. RECOMENDACIONES CLAVE:
   a) COMENZAR CON FASE 1 inmediatamente
   b) ASIGNAR RECURSOS específicos para cada propuesta
   c) ESTABLECER MÉTRICAS de seguimiento desde el inicio
   d) COMUNICAR EL PLAN a todos los stakeholders
   e) REVISAR PROGRESO mensualmente

7. PRÓXIMOS PASOS INMEDIATOS:
   1. Aprobación del plan por gerencia
   2. Asignación de presupuesto inicial
   3. Formación de equipo de implementación
   4. Desarrollo de cronograma detallado
   5. Comunicación oficial del proyecto

{'='*100}
INVERSIÓN TOTAL ESTIMADA: USD {presupuesto_por_fase['Inversion_Estimada_USD'].sum():,.0f}
ROI TOTAL ESPERADO: {presupuesto_por_fase['Inversion_Estimada_USD'].sum() * presupuesto_por_fase['ROI_Estimado'].mean() / presupuesto_por_fase['Inversion_Estimada_USD'].sum():.1f}x
PERIODO DE RECUPERACIÓN: {12 / presupuesto_por_fase['ROI_Estimado'].mean():.1f} meses
{'='*100}

✅ EJERCICIO COMPLETADO - RECOMENDACIONES LISTAS PARA IMPLEMENTACIÓN
{'='*100}
"""

# Guardar reporte final
with open(f'{OUTPUT_DIR}/Reporte_Final_Propuestas_Mejora.txt', 'w', encoding='utf-8') as f:
    f.write(reporte_final)

print(f"   ✅ Archivos guardados en '{OUTPUT_DIR}/':")
print(f"      1. Propuestas_Mejora_Completas.csv")
print(f"      2. Plan_Accion_Ejecutivo.csv")
print(f"      3. Resumen_por_Categoria.csv")
print(f"      4. Reporte_Final_Propuestas_Mejora.txt")
print(f"      5. Roadmap_Propuestas_Mejora.png")

# ===============================
# 8. MOSTRAR RESUMEN EJECUTIVO
# ===============================
print("\n" + "="*100)
print("📋 RESUMEN EJECUTIVO - PROPUESTAS DE MEJORA")
print("="*100)

print(f"""
🎯 RESULTADOS DEL EJERCICIO FIFO FÍSICO:
   • Residuos analizados: {total_residuos}
   • Residuos ajustados: {residuos_ajustados} ({porcentaje_ajustados:.1f}%)
   • Error corregido: {df_resumen_ejecutivo['Error_Total_Corregido_kg'].iloc[0]:,.0f} kg
   • Ajustes aplicados: {len(df_log_cambios)}

💡 PROPUESTAS GENERADAS: {len(df_propuestas)}
   Categorías:
   • Tecnológicas: {len(df_propuestas[df_propuestas['Categoria'] == 'TECNOLÓGICA'])}
   • Procesos: {len(df_propuestas[df_propuestas['Categoria'] == 'PROCESOS'])}
   • Capacitación: {len(df_propuestas[df_propuestas['Categoria'] == 'CAPACITACIÓN'])}
   • Monitoreo: {len(df_propuestas[df_propuestas['Categoria'] == 'MONITOREO'])}

💰 INVERSIÓN Y RETORNO:
   • Inversión total estimada: USD {presupuesto_por_fase['Inversion_Estimada_USD'].sum():,.0f}
   • ROI promedio esperado: {presupuesto_por_fase['ROI_Estimado'].mean():.1f}x
   • Período de recuperación: {12 / presupuesto_por_fase['ROI_Estimado'].mean():.1f} meses

📅 ROADMAP DE IMPLEMENTACIÓN:
   • FASE 1 (0-3 meses): {len(fase1_propuestas)} propuestas prioritarias
   • FASE 2 (3-6 meses): {len(df_propuestas[df_propuestas['Fase_Implementacion'] == 'FASE 2: Mediano Plazo (3-6 meses)'])} propuestas
   • FASE 3 (6-12 meses): {len(df_propuestas[df_propuestas['Fase_Implementacion'] == 'FASE 3: Largo Plazo (6-12 meses)'])} propuestas

🏆 TOP 3 PROPUESTAS PRIORITARIAS:
""")

for idx, row in df_propuestas.head(3).iterrows():
    print(f"   {row['Ranking']}. {row['Propuesta']}")
    print(f"      {row['Descripcion'][:60]}...")
    print(f"      Inversión: USD {row['Inversion_Estimada_USD']:,.0f} | ROI: {row['ROI_Estimado']:.1f}x")
    print(f"      Responsable: {row['Responsable']}")
    print()

print("🚀 PRÓXIMOS PASOS RECOMENDADOS:")
print("   1. Revisar y aprobar el plan de acción ejecutivo")
print("   2. Asignar presupuesto para Fase 1")
print("   3. Designar equipo de implementación")
print("   4. Establecer sistema de seguimiento de métricas")
print("   5. Programar primera revisión de progreso en 30 días")

print("\n" + "="*100)
print("✅ EJERCICIO COMPLETADO EXITOSAMENTE")
print("="*100)
print("""
🎯 LOGRADO EN ESTE EJERCICIO:

1. GENERACIÓN DE DATOS SINTÉTICOS REALISTAS:
   • Escenario con problemas comunes de balance
   • Errores diversos para simular condiciones reales

2. ANÁLISIS COMPLETO CON MÉTODO FIFO FÍSICO:
   • Identificación sistemática de discrepancias
   • Corrección precisa de errores
   • Validación de resultados

3. DASHBOARD DE VISUALIZACIÓN:
   • Gráficos completos de resultados
   • Métricas clave de desempeño
   • Reportes ejecutivos

4. PROPUESTAS DE MEJORA BASADAS EN DATOS:
   • Recomendaciones específicas y cuantificadas
   • ROI estimado para cada propuesta
   • Plan de implementación por fases

📁 ARCHIVOS FINALES GENERADOS:
   • 15+ archivos de datos, resultados y visualizaciones
   • Dashboard completo en PNG y PDF
   • Plan de acción ejecutivo priorizado
   • Reportes detallados en texto y CSV

🔮 BENEFICIOS ESPERADOS DE LA IMPLEMENTACIÓN:
   • Reducción del 90% en errores de registro
   • Eliminación de balances negativos
   • Mejora del 80% en trazabilidad
   • Ahorro anual significativo en costos
   • Mejor cumplimiento normativo

🌟 ¡FELICITACIONES! Ha completado exitosamente el ejercicio completo
   de aplicación del método FIFO Físico para gestión de residuos.
""")
print("="*100)

# Mostrar gráfico
plt.show()